# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Notebook authored and statically validated but NOT executed: every colab exec failed with WebSocketConnectionClosedException on the first kernel send across 2 fresh CPU sessions in 2 regions; ~/.config/colab-cli/colab.log shows all kernels created since 2026-10-23 ~02:11 UTC stuck in execution_state 'starting' (also for other workers' sessions) followed by [Errno 110] Connection timed out. Systemic Colab-backend/CLI-0.7.5 execution-environment failure, not a scientific missing-material conclusion. Notebook preserved at deliverables/p-c36d1f29a831a7708295479260406a78.ipynb for the unverified-reference path; retry via resume_run_dir after backend recovery (probe: logs/probe_layout.ipynb).

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Biology-aware association for identity-consistent leaf tracking in top-down plant phenotyping

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-c36d1f29a831a7708295479260406a78)


# Biology-aware association for identity-consistent leaf tracking — minimal reproduction

**Paper:** Dongze Zhou, Cong Zhang, *"Biology-aware association for identity-consistent leaf tracking in top-down plant phenotyping"*, Front. Plant Sci. 17 (2026), DOI [10.3389/fpls.2026.1881413](https://doi.org/10.3389/fpls.2026.1881413)

**Author code:** [`awwbugbug/bioaware-leaf-tracking`](https://github.com/awwbugbug/bioaware-leaf-tracking) pinned at `5a9cfbfe7cb5be042910a1ef1444118f01df8f75`

**Data:** [shl-shawn/CanolaTrack](https://huggingface.co/datasets/shl-shawn/CanolaTrack) (public; © BASF SE 2025, free for non-commercial research and education)

## What this notebook does — PARTIAL DEMONSTRATION (3 of 37 validation plants)

The paper's contribution is the **association stage** of a tracking-by-detection pipeline: a Hungarian
assignment whose cost blends one appearance cue with three biology/geometric cues (spatial continuity,
area-growth plausibility, lifecycle plausibility). All compared methods share the same detections, so
only the association stage changes (paper §2.2–2.3).

This notebook reproduces that comparison on a **deterministic 3-plant subset** (Plant-148/149/150, the
first three plants of the 37-plant CanolaTrack validation split):

1. **Centroid+Hungarian** and **IoU+Hungarian** naive baselines (author `trackers/`).
2. **Fixed-weight bio-aware tracker** — paper Table 1 row "Fixed bio-constraints" (λ_app=1.0, λ_pos=0.3,
   λ_area=0.2, λ_life=0.1, τ_app=0.4, coast 5 frames) using author `BioTracker` + released LeafTrackNet
   ReID checkpoint.
3. Both are scored with the author's evaluation stack (**TrackEval**, HOTA / CLEAR / Identity) and
   compared *in direction* with the paper's full-validation reference (Centroid IDSW 438 / HOTA 67.99;
   IoU 395 / 77.50; Fixed bio 147 / 89.04).

**Out of scope / not reproduced:** the adaptive Weight-Predictor variant (the headline row of the paper)
because no checkpoint is released and training uses the 147-plant training split; the YOLOv10 detector
(shared pre-computed proposals `det_db_val.json` are used instead); the full 37-plant benchmark; the
external GT-box and perturbation studies. A 3-plant subset demonstrates the mechanism and the direction
of the effect — it cannot verify dataset-level accuracy.

**日本語:** 本ノートブックは、CanolaTrack 検証分割の先頭 3 植物で、論文の「biology 制約付き関連付け」
を最小規模に再現します。Centroid / IoU の単純ベースラインと固定重み BioTracker（著者コード＋公開
ReID 重み）を TrackEval（HOTA / CLEAR / Identity）で比較します。適応型 Weight Predictor（チェック
ポイント未公開・学習が必要）と YOLOv10 検出器（共有提案を使用）は範囲外です。

## Runtime selection & Run all

- **Runtime → Change runtime type → CPU** (standard) is sufficient. The workload is small Hungarian
  assignments plus MobileNetV3 crop embeddings (~93 frames × ~10 crops each). A GPU runtime also works
  without changes (the author code selects CUDA when available) but is **not required**.
- Expected wall time after *Run all*: ~5–10 minutes; downloads ≈ 35 MB.

**日本語:** ランタイムは CPU で十分です（GPU でもそのまま動作）。Run all で約 5〜10 分、ダウンロード約 35 MB。

### Dependencies

The author's `requirements.txt` lists `torch, torchvision, numpy, scipy, Pillow, PyYAML, tqdm,
pycocotools`. Colab preinstalls the heavy scientific stack; only `PyYAML` and `tqdm` are needed for the
association path (`pycocotools` is used only by the author's training utilities, which this notebook
does not run).

In [ ]:
%pip install -q PyYAML tqdm

import sys
import numpy, scipy, PIL, torch, torchvision
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("numpy       :", numpy.__version__)
print("scipy       :", scipy.__version__)

### Pin the author code and the evaluation stack

Both repositories are cloned **inside Colab** and checked out at exact commits (asserted):

- `awwbugbug/bioaware-leaf-tracking` @ `5a9cfbfe7cb5be042910a1ef1444118f01df8f75` — the paper's official implementation (master head).
- `JonathonLuiten/TrackEval` @ `12c8791b303e0a0b50f753af204249e622d0281a` — the evaluation stack the author's `scripts/eval.sh` calls.

**日本語:** 著者コードと TrackEval を、コミットを明示してピン留した状態で Colab 内にクローンします。

In [ ]:
import os, subprocess

os.chdir("/content")
AUTHOR_URL = "https://github.com/awwbugbug/bioaware-leaf-tracking.git"
AUTHOR_SHA = "5a9cfbfe7cb5be042910a1ef1444118f01df8f75"
TRACKEVAL_URL = "https://github.com/JonathonLuiten/TrackEval.git"
TRACKEVAL_SHA = "12c8791b303e0a0b50f753af204249e622d0281a"


def pinned_clone(url, sha, dest):
    if not os.path.exists(dest):
        subprocess.run(["git", "clone", "--quiet", "--filter=blob:none", url, dest], check=True)
    subprocess.run(["git", "-C", dest, "checkout", "--quiet", "--detach", sha], check=True)
    head = subprocess.run(["git", "-C", dest, "rev-parse", "HEAD"], check=True,
                          capture_output=True, text=True).stdout.strip()
    assert head == sha, f"{dest}: HEAD {head} != pinned {sha}"
    print(f"{dest} pinned at {head[:12]}")


pinned_clone(AUTHOR_URL, AUTHOR_SHA, "bioaware-leaf-tracking")
pinned_clone(TRACKEVAL_URL, TRACKEVAL_SHA, "TrackEval")

### Download the minimal CanolaTrack inputs (Colab only)

From the public, non-gated HF dataset `shl-shawn/CanolaTrack` this fetches exactly what the
demonstration needs: shared validation proposals, the released ReID checkpoint, and the 3 selected
plants' `gt/gt.txt` + `img/*.jpg` (~31 daily frames per plant, 1200×1200 px).

**日本語:** 共有検出提案・ReID 重み・選択した 3 植物の画像と正解トラックのみを Colab 内にダウンロードします。

In [ ]:
import json, time, urllib.request
from pathlib import Path

HF_BASE = "https://huggingface.co/datasets/shl-shawn/CanolaTrack/resolve/main/"
API_BASE = "https://huggingface.co/api/datasets/shl-shawn/CanolaTrack/tree/main/"
ROOT = Path("/content/CanolaTrack")
PLANTS = ["Plant-148", "Plant-149", "Plant-150"]  # first three of the 37 val plants (deterministic)


def fetch(rel, dest, tries=3):
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(HF_BASE + rel, timeout=120) as r:
                data = r.read()
            dest.write_bytes(data)
            return len(data)
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))


n = fetch("proposals/det_db_val.json", ROOT / "proposals/det_db_val.json")
print("det_db_val.json     :", n, "bytes")
n = fetch("weights/LeafTrackNet_CanolaTrack.pth", ROOT / "weights/LeafTrackNet_CanolaTrack.pth")
print("LeafTrackNet weights:", n, "bytes")

for plant in PLANTS:
    n = fetch(f"CanolaTrack/val/{plant}/gt/gt.txt", ROOT / f"CanolaTrack/val/{plant}/gt/gt.txt")
    listing = json.load(urllib.request.urlopen(API_BASE + f"CanolaTrack/val/{plant}/img", timeout=60))
    for entry in listing:
        name = entry["path"].split("/")[-1]
        fetch(entry["path"], ROOT / f"CanolaTrack/val/{plant}/img/{name}")
    print(plant, ": gt.txt", n, "bytes,", len(listing), "images")

### Verify the downloaded bytes and data layout

The pinned investigation could only read metadata, so the actual payloads are checked here:
proposal JSON key format (`val/<plant>/<frame>.txt`), checkpoint tensor shapes, ground-truth file
columns (TrackEval expects `frame, id, x, y, w, h, mark, class, …`), and image dimensions.

**日本語:** ダウンロードした実データ（提案キーの形式・チェックポイントの形状・GT の列構成・画像サイズ）を検証します。

In [ ]:
import collections
import json
import torch
from PIL import Image

det = json.load(open(ROOT / "proposals/det_db_val.json"))
keys = list(det)
print("proposal entries:", len(keys), "| example key:", keys[0])
assert all(len(k.split("/")) >= 3 for k in keys), "unexpected proposal key format"

plants_all = sorted({k.split("/")[-2] for k in keys})
print("plants covered by proposals:", len(plants_all))
assert set(PLANTS).issubset(plants_all), "selected plants missing from proposals"

frames = {}  # 1-based daily frame index per plant, from the proposal keys
for plant in PLANTS:
    fs = sorted(int(k.split("/")[-1].split(".")[0]) for k in keys if k.split("/")[-2] == plant)
    frames[plant] = fs
    print(plant, "frames:", min(fs), "..", max(fs), "count:", len(fs))

ckpt = torch.load(str(ROOT / "weights/LeafTrackNet_CanolaTrack.pth"),
                  map_location="cpu", weights_only=True)
print("checkpoint tensors:", len(ckpt))
for k in list(ckpt)[:4]:
    print("  ", k, tuple(ckpt[k].shape))

gt_lines = (ROOT / "CanolaTrack/val/Plant-148/gt/gt.txt").read_text().strip().splitlines()
print("gt.txt head:", gt_lines[:2])
ncols = collections.Counter(len(l.split(",")) for l in gt_lines)
print("gt column counts:", dict(ncols))
assert max(ncols) >= 8, "GT has fewer than 8 columns; TrackEval format check needed"
cls = collections.Counter(l.split(",")[7] for l in gt_lines)
print("gt class ids (col 8):", dict(cls))
assert set(cls) == {"1"}, "GT class column is not all '1' (TrackEval 'pedestrian') — adapt eval"

img = Image.open(ROOT / f"CanolaTrack/val/Plant-148/img/{min(frames['Plant-148']):08d}.jpg")
print("image size (W, H):", img.size)

### Input preview — proposals vs ground truth

The first validation frame of Plant-148 with the shared proposals (green) and the ground-truth leaf
boxes of the same frame (red). These are the actual tracker inputs; overlapping neighbouring leaves
are exactly the case where appearance and biology cues must disambiguate.

**日本語:** 実際の入力フレームに、共有提案（緑）と正解ボックス（赤）を重ねて表示します。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

plant = "Plant-148"
frame = min(frames[plant])
img = Image.open(ROOT / f"CanolaTrack/val/{plant}/img/{frame:08d}.jpg").convert("RGB")

prop_boxes = []
for ln in det[f"val/{plant}/{frame:08d}.txt"]:
    parts = ln.strip().split(",")
    if len(parts) >= 5:
        x, y, w, h, conf = map(float, parts[:5])
        prop_boxes.append((x, y, w, h, conf))

gt_boxes = []
for ln in (ROOT / f"CanolaTrack/val/{plant}/gt/gt.txt").read_text().strip().splitlines():
    p = ln.split(",")
    if int(float(p[0])) == frame:
        gt_boxes.append(tuple(map(float, p[2:6])))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img)
for i, (x, y, w, h, conf) in enumerate(prop_boxes):
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="lime", lw=1.4))
    ax.text(x, y - 4, f"P{i+1}", color="lime", fontsize=7)
for x, y, w, h in gt_boxes:
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", lw=1.4, linestyle="--"))
ax.add_patch(patches.Rectangle((0, 0), 0, 0, edgecolor="lime", label="shared proposals (det_db_val)"))
ax.add_patch(patches.Rectangle((0, 0), 0, 0, edgecolor="red", linestyle="--", label="ground truth (gt.txt)"))
ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"{plant} frame {frame} - {len(prop_boxes)} proposals vs {len(gt_boxes)} GT boxes")
ax.axis("off")
plt.show()

### Run the three trackers on the selected plants

This uses the author's pinned code unchanged: `CentroidTracker(max_age=5, max_dist=200)` and
`IoUTracker(max_age=5, min_iou=0.1)` (author `run_eval/` defaults), and `BioTracker` with the paper's
fixed constraint weights (the `run_bio.py` defaults). A fresh tracker instance is used per plant,
exactly as in `run_bio.py`; the only adaptation is that we loop over the 3 selected plants instead of
all 37. Outputs are MOT-format `<frame>, <id>, <x>, <y>, <w>, <h>, <conf>, -1, -1, -1` files per plant.

**日本語:** 著者コードのデフォルト設定のまま、3 植物に各トラッカーを実行します（BioTracker
は公開 ReID 重みで葉クロップを埋め込み、固定重み λ=1.0/0.3/0.2/0.1 を使用）。

In [ ]:
import sys, time
sys.path.insert(0, "/content/bioaware-leaf-tracking")

import torch, yaml
from PIL import Image

from models import LeafReIDModel
from utils.io import key_to_plant_frame, parse_detection_lines, write_mot
from utils.transforms import get_infer_transform
from trackers.bio_tracker import BioTracker
from trackers.centroid_tracker import CentroidTracker
from trackers.iou_tracker import IoUTracker

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open("/content/bioaware-leaf-tracking/configs/default.yaml"))
reid = LeafReIDModel(cfg["backbone"], embed_dim=cfg["embed_dim"], pretrained=False).to(device)
reid.load_state_dict(torch.load(str(ROOT / "weights/LeafTrackNet_CanolaTrack.pth"),
                                map_location=device))
reid.eval()
print("ReID checkpoint loaded on", device)


def plant_keys(plant):
    return sorted((k for k in keys if k.split("/")[-2] == plant),
                  key=lambda k: int(k.split("/")[-1].split(".")[0]))


def write_tracks(name, make_tracker, use_images):
    out = Path(f"/content/outputs/{name}/tracks")
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for plant in PLANTS:
        tracker = make_tracker()
        lines = []
        for k in plant_keys(plant):
            boxes, _ = parse_detection_lines(det[k])
            _, frame = key_to_plant_frame(k)
            if use_images:
                img_path = ROOT / f"CanolaTrack/val/{plant}/img/{frame:08d}.jpg"
                if not img_path.exists():
                    continue  # same guard as author run_bio.py
                img = Image.open(img_path).convert("RGB")
                ids, sims = tracker.update(img, boxes)
            else:
                ids, sims = tracker.update(boxes)
            for (x1, y1, x2, y2), tid, s in zip(boxes, ids, sims):
                lines.append(f"{frame}, {int(tid)}, {x1:.2f}, {y1:.2f}, "
                             f"{x2 - x1:.2f}, {y2 - y1:.2f}, {float(s):.4f}, -1, -1, -1")
        write_mot(str(out / f"{plant}.txt"), lines)
    return time.time() - t0


timings = {}
timings["centroid"] = write_tracks("centroid", lambda: CentroidTracker(max_age=5, max_dist=200), False)
timings["iou"] = write_tracks("iou", lambda: IoUTracker(max_age=5, min_iou=0.1), False)
timings["bio"] = write_tracks(
    "bio",
    lambda: BioTracker(
        reid_model=reid, transform=get_infer_transform(), device=str(device),
        similarity_threshold=0.4, max_age=5, update_mode="mean", alpha=0.5,
        lambda_app=1.0, lambda_pos=0.3, lambda_area=0.2, lambda_life=0.1,
        max_norm_dist=0.25, area_shrink_tol=0.30, area_penalty_scale=1.0,
    ), True)
for name, t in timings.items():
    print(f"{name:8s}: {t:.1f} s  -> /content/outputs/{name}/tracks/")

### Evaluate with TrackEval (the author's metric stack)

The author's `scripts/eval.sh` calls `TrackEval/scripts/run_mot_challenge.py` with
`--GT_FOLDER .../val --SEQMAP_FILE seqmap --SKIP_SPLIT_FOL True`. Two tiny AI-generated adapters
(documented glue, not scientific changes) are needed because the public dataset ships no
`seqinfo.ini`/seqmap: (1) a seqmap listing our 3 sequences, and (2) a `seqinfo.ini` per plant with the
frame count taken from the verified proposal keys (TrackEval uses 1-based frame indices, which the
ground-truth files already use).

**日本語:** TrackEval は `seqinfo.ini` と seqmap を要求するため、検証済みのフレーム数から
最小限のアダプタ（AI 生成の補助コード）を生成して評価します。

In [ ]:
import subprocess

# --- AI-generated adapters (documented): seqmap + seqinfo.ini -----------------
Path("/content/seqmap.txt").write_text("name\n" + "\n".join(PLANTS) + "\n")
for plant in PLANTS:
    seq_len = max(frames[plant])
    (ROOT / f"CanolaTrack/val/{plant}/seqinfo.ini").write_text(
        f"[Sequence]\nname={plant}\nimDir=img\nseqLength={seq_len}\n")
print("wrote seqmap.txt and seqinfo.ini for", PLANTS)

# --- TrackEval: HOTA / CLEAR / Identity on the 3 sequences -------------------
cmd = [sys.executable, "/content/TrackEval/scripts/run_mot_challenge.py",
       "--SPLIT_TO_EVAL", "val", "--METRICS", "HOTA", "CLEAR", "Identity",
       "--GT_FOLDER", "/content/CanolaTrack/val/",
       "--SEQMAP_FILE", "/content/seqmap.txt", "--SKIP_SPLIT_FOL", "True",
       "--TRACKERS_FOLDER", "/content/outputs",
       "--TRACKERS_TO_EVAL", "centroid", "iou", "bio",
       "--USE_PARALLEL", "False", "--PLOT_CURVES", "False",
       "--OUTPUT_SUMMARY", "True", "--OUTPUT_FOLDER", "/content/outputs/eval"]
res = subprocess.run(cmd, capture_output=True, text=True)
print("TrackEval exit code:", res.returncode)
print(res.stdout[-2500:])
if res.returncode != 0:
    print(res.stderr[-2000:])
    raise RuntimeError("TrackEval failed - see stderr above")

### Results — subset metrics vs the paper's full-validation reference

Each `<tracker>/pedestrian_summary.txt` holds the combined-sequence metrics. We tabulate HOTA, AssA
(association accuracy) and IDSW per method and plot IDSW. Absolute values on 3 plants differ from the
paper's 37-plant benchmark; the reproduced *direction* (bio-aware ≤ naive baselines in IDSW) is the
check this notebook supports.

**日本語:** 各トラッカーの COMBINED_SEQ 結果を表と IDSW グラフにします。3 植物のサブセットなので
絶対値は論文の 37 植物の値とは異なります。

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd

rows = {}
for f in sorted(glob.glob("/content/outputs/eval/*/*_summary.txt")):
    tracker = Path(f).parent.name
    lines = Path(f).read_text().strip().splitlines()
    header, values = lines[0].split(), lines[1].split()
    rows[tracker] = dict(zip(header, values))

results = pd.DataFrame(rows).T
show = [c for c in ["HOTA", "DetA", "AssA", "MOTA", "IDF1", "IDSW"] if c in results.columns]
results[show] = results[show].apply(pd.to_numeric, errors="coerce")
display(results[show])
results[show].to_csv("/content/bio_association_subset_results.csv", index_label="method")
print("saved /content/bio_association_subset_results.csv")

paper_ref = pd.DataFrame(
    {"IDSW": [438, 395, 147], "HOTA": [67.99, 77.50, 89.04]},
    index=["centroid (paper, 37 plants)", "iou (paper, 37 plants)", "bio fixed (paper, 37 plants)"])
print("Paper reference (full validation, Table 1):")
display(paper_ref)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
results[["IDSW"]].plot.bar(ax=axes[0], legend=False, color="#4c78a8")
axes[0].set_title("IDSW on 3-plant subset (lower is better)")
axes[0].set_ylabel("identity switches")
results[["HOTA", "AssA"]].plot.bar(ax=axes[1])
axes[1].set_title("HOTA / AssA on 3-plant subset (higher is better)")
axes[1].set_ylim(0, 100)
axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## Interpretation & validation record

- **What was executed:** the authors' pinned association code on 3 validation plants with shared
  proposals and the released ReID checkpoint, scored by TrackEval (the author's own metric stack).
- **What to check in the outputs above:** whether the fixed-weight bio-aware tracker reduces IDSW
  relative to the centroid/IoU baselines on this subset, matching the direction of the paper's full
  benchmark (Centroid 438 → Fixed bio 147 on 37 plants). Subset HOTA/AssA values are not comparable
  in absolute terms to the paper's Table 1.
- **Known limitations:** 3-plant subset (not a dataset-level verification); adaptive Weight-Predictor
  variant not run (no released checkpoint; would require training on the 147-plant training split);
  detection quality is inherited from the shared proposals; AI-generated glue is limited to
  `seqinfo.ini`/seqmap generation and the per-plant loop — all tracker/evaluator code is the authors'
  pinned implementation.
- **Provenance:** paper DOI 10.3389/fpls.2026.1881413 (Front. Plant Sci. 17, 2026-10-14); author repo
  `awwbugbug/bioaware-leaf-tracking` @ `5a9cfbfe7cb5be042910a1ef1444118f01df8f75`; data `shl-shawn/CanolaTrack` (public HF dataset,
  © BASF SE 2025, non-commercial research/education use); TrackEval @ `12c8791b303e0a0b50f753af204249e622d0281a`.

**日本語:** 本ノートブックは 3 植物の部分再現です。固定重み BioTracker が IDSW を
Centroid/IoU ベースラインより削減するか（論文の方向性と整合するか）を確認してください。
絶対値は論文の 37 植物ベンチマークとは比較できません。適応型（学習が必要）は範囲外です。